# CCHP Kaggle 入口 Notebook（main.ipynb）

目标：在 Kaggle Notebook 上直接运行本仓库 `src/cchp_physical_env` 的代码，并使用你上传的数据集 `data/processed/*.csv`。

使用方法（Kaggle 右侧 Add data）：

1. 添加一个“代码数据集”（建议命名类似 `cchp-code`），其顶层包含：

- `pyproject.toml`
- `src/`

2. 添加一个“数据数据集”（建议命名类似 `cchp-data`），其顶层包含：

- `data/processed/cchp_main_15min_2024.csv`
- `data/processed/cchp_main_15min_2025.csv`

运行说明：

- Kaggle 的 `/kaggle/input/...` 是只读的。本 notebook 会把代码和数据 rsync 到当前工作目录（可写），再执行。
- 默认 CLI 会读取：
  - `data/processed/cchp_main_15min_2024.csv`
  - `data/processed/cchp_main_15min_2025.csv`
- 你只需要按顺序运行每个 cell。


In [ ]:
# 1) 路径自检：建议先运行这一格
from pathlib import Path

print('cwd =', Path.cwd())
print('data/processed exists =', Path('data/processed').exists())
print('processed csv =', [p.name for p in Path('data/processed').glob('*.csv')])


In [ ]:
from __future__ import annotations

from pathlib import Path
import os
import sys

print('Python:', sys.version)
print('CWD:', Path.cwd())
print('Has /kaggle/input:', Path('/kaggle/input').exists())
print('Has /kaggle/working:', Path('/kaggle/working').exists())
print('CPU count:', os.cpu_count())

for k in ['OMP_NUM_THREADS', 'MKL_NUM_THREADS', 'OPENBLAS_NUM_THREADS', 'NUMEXPR_NUM_THREADS']:
    if k in os.environ:
        print(f'{k}:', os.environ.get(k))


## 1) 同步代码到可写目录并安装

说明：

- Kaggle 的 `/kaggle/input/...` 是只读。
- 我们把代码同步到当前工作目录（可写）后，再执行 `%pip install -e . --no-deps`。
- 为避免 Kaggle 的 editable 安装在某些镜像下没有把 `./src` 自动加入 `sys.path`，这里会显式插入 `./src`。


In [ ]:
from pathlib import Path

def _find_codeset_dir(preferred: str = '/kaggle/input/cchp-code') -> Path:
    p = Path(preferred)
    if (p / 'pyproject.toml').exists() and (p / 'src').exists():
        return p

    root = Path('/kaggle/input')
    if not root.exists():
        return p

    hits: list[Path] = []
    for pyproj in root.rglob('pyproject.toml'):
        cand = pyproj.parent
        if (cand / 'src' / 'cchp_physical_env').exists():
            hits.append(cand)

    if len(hits) == 1:
        return hits[0]
    if len(hits) > 1:
        hits_sorted = sorted(hits, key=lambda x: x.as_posix())
        print('Multiple code datasets detected. Using:', hits_sorted[0])
        return hits_sorted[0]

    return p


code_src = _find_codeset_dir()
assert code_src.exists(), f'missing code dataset: {code_src}'
print('code_src:', code_src)

# 同步代码（顶层应包含 pyproject.toml 与 src/）
!rsync -a --info=progress2 {code_src.as_posix()}/ ./

assert Path('pyproject.toml').exists(), 'missing pyproject.toml after sync'
assert Path('src/cchp_physical_env').exists(), 'missing src/cchp_physical_env after sync'

# 兜底：确保可 import
src_dir = str((Path.cwd() / 'src').resolve())
if src_dir not in sys.path:
    sys.path.insert(0, src_dir)
print('sys.executable:', sys.executable)
print('sys.path[0]:', sys.path[0])

# 安装依赖（装到当前 kernel 环境）
%pip -q install -e . --no-deps

# 核心依赖兜底（部分 Kaggle 镜像可能未预装）
%pip -q install numpy pandas pyyaml

import cchp_physical_env
print('cchp_physical_env imported from:', cchp_physical_env.__file__)


## 2) 同步数据到 `./data/`

说明：

- 你的 Kaggle 数据集顶层应包含 `data/processed/`。
- 本 cell 会把 `.../data/` 同步到当前工作目录的 `./data/`，以符合仓库 CLI 的默认路径。


In [ ]:
from pathlib import Path

def _find_data_root(preferred: str = '/kaggle/input/cchp-data') -> Path:
    p = Path(preferred)
    if (p / 'data' / 'processed').exists():
        return p

    root = Path('/kaggle/input')
    if not root.exists():
        return p

    hits: list[Path] = []
    for d in root.rglob('data'):
        cand = d.parent
        if d.is_dir() and (d / 'processed').exists():
            # 避免误判：至少要有一个 processed csv
            if list((d / 'processed').glob('*.csv')):
                hits.append(cand)

    if len(hits) == 1:
        return hits[0]
    if len(hits) > 1:
        hits_sorted = sorted(hits, key=lambda x: x.as_posix())
        print('Multiple data datasets detected. Using:', hits_sorted[0])
        return hits_sorted[0]

    return p


data_root = _find_data_root()
assert data_root.exists(), f'missing data dataset: {data_root}'
print('data_root:', data_root)

Path('data').mkdir(parents=True, exist_ok=True)
!rsync -a --info=progress2 {data_root.as_posix()}/data/ data/

print('data/processed exists:', Path('data/processed').exists())
print('processed csv =', [p.name for p in Path('data/processed').glob('*.csv')])

# 快速检查关键文件
train_csv = Path('data/processed/cchp_main_15min_2024.csv')
eval_csv = Path('data/processed/cchp_main_15min_2025.csv')
print('train_csv exists:', train_csv.exists(), train_csv)
print('eval_csv exists:', eval_csv.exists(), eval_csv)


## 3) 运行仓库 CLI 自检（summary / train / eval）

说明：

- `summary`：读取 2024/2025 数据，做冻结 schema 一致性校验，并输出摘要。
- `train`：跑一个最小训练（baseline rule，默认读取 `src/cchp_physical_env/config/config.yaml`）。
- `eval`：用训练产物中的 checkpoint 跑评估。

注意：

- Kaggle 环境如果没有外部 solver（如 glpk），请保持 `config.yaml` 中的 `pyomo_solver: projection`。
- `runs/` 会写到当前工作目录（Kaggle 可下载 output）。


In [ ]:
import sys
from pathlib import Path

env_cfg = Path('src/cchp_physical_env/config/config.yaml')
assert env_cfg.exists(), env_cfg

# 1) summary
!{sys.executable} -m cchp_physical_env summary --env-config {env_cfg.as_posix()}


In [ ]:
# 2) 最小 train（生成 runs/.../checkpoints/baseline_policy.json）
# 你可以按需要改 seed / episodes / episode-days
!{sys.executable} -m cchp_physical_env train --env-config {env_cfg.as_posix()} --episodes 1 --episode-days 7 --policy rule --seed 2


In [ ]:
# 3) eval：从刚才 train 的 runs 目录中自动找到 baseline_policy.json 再评估
import re

def _latest_run_dir(run_root: Path = Path('runs')) -> Path:
    if not run_root.exists():
        raise FileNotFoundError('runs/ not found. Please run train cell first.')
    candidates = [p for p in run_root.iterdir() if p.is_dir()]
    if not candidates:
        raise FileNotFoundError('No run directories under runs/.')
    # 目录名形如 20260304_185358_train_rule；按名字排序即可
    return sorted(candidates, key=lambda p: p.name)[-1]

run_dir = _latest_run_dir()
ckpt = run_dir / 'checkpoints' / 'baseline_policy.json'
assert ckpt.exists(), ckpt
print('Using checkpoint:', ckpt)

!{sys.executable} -m cchp_physical_env eval --env-config {env_cfg.as_posix()} --checkpoint {ckpt.as_posix()} --seed 2


## 4) Kaggle 运行注意事项（常见坑）

- 代码/数据都必须通过 Add data 挂载到 `/kaggle/input/...`，并且 **不要在只读目录里直接运行**（会写不出 `runs/`）。
- 如果 `data/processed` 里缺少 `cchp_main_15min_2025.csv`，`summary` 会失败（会做 2024/2025 的冻结一致性校验）。
- 若遇到求解器不可用：
  - 保持 `src/cchp_physical_env/config/config.yaml` 中 `pyomo_solver: projection`（无需外部 solver）。
- 输出产物：
  - 训练与评估结果默认写入 `./runs/`（Kaggle 会保存在当前 Session 的 working 目录，可通过右侧 Output 下载）。
